In [28]:
import os
import random
from utils.paths import path_data_cholec_split, path_data_cholec_original, path_data_cholec, path_data_cholect50
seed = 42

In [29]:
videos_cholec = os.listdir(os.path.join(path_data_cholec, "video_frames"))
videos_cholec_seg = [f for f in os.listdir(path_data_cholec_original) if os.path.isdir(os.path.join(path_data_cholec_original, f))]
videos_cholec_scene_graph = os.listdir(os.path.join(path_data_cholect50, "videos"))

videos_cholec = sorted([int(v[len("video"):]) for v in videos_cholec])
videos_cholec_seg = sorted([int(v[len("video"):]) for v in videos_cholec_seg])
videos_cholec_scene_graph = sorted([int(v[len("VID"):]) for v in videos_cholec_scene_graph])

In [30]:
all_videos = set(videos_cholec).union(set(videos_cholec_seg)).union(set(videos_cholec_scene_graph))
video_annotations = {v: [] for v in all_videos}
for v in videos_cholec:
    video_annotations[v].append("pseudo-segmentation")
for v in videos_cholec_seg:
    video_annotations[v].append("segmentation")
for v in videos_cholec_scene_graph:
    video_annotations[v].append("scene-graph")

In [31]:
num_videos = len(all_videos)
video_annotations

{1: ['pseudo-segmentation', 'segmentation', 'scene-graph'],
 2: ['pseudo-segmentation', 'scene-graph'],
 3: ['pseudo-segmentation'],
 4: ['pseudo-segmentation', 'scene-graph'],
 5: ['pseudo-segmentation', 'scene-graph'],
 6: ['pseudo-segmentation', 'scene-graph'],
 7: ['pseudo-segmentation'],
 8: ['pseudo-segmentation', 'scene-graph'],
 9: ['pseudo-segmentation', 'segmentation'],
 10: ['pseudo-segmentation', 'scene-graph'],
 11: ['pseudo-segmentation'],
 12: ['pseudo-segmentation', 'segmentation', 'scene-graph'],
 13: ['pseudo-segmentation', 'scene-graph'],
 14: ['pseudo-segmentation', 'scene-graph'],
 15: ['pseudo-segmentation', 'scene-graph'],
 16: ['pseudo-segmentation'],
 17: ['pseudo-segmentation', 'segmentation'],
 18: ['pseudo-segmentation', 'segmentation', 'scene-graph'],
 19: ['pseudo-segmentation'],
 20: ['pseudo-segmentation', 'segmentation'],
 21: ['pseudo-segmentation'],
 22: ['pseudo-segmentation', 'scene-graph'],
 23: ['pseudo-segmentation', 'scene-graph'],
 24: ['pseudo

In [32]:
train_cases = set()
test_val_cases = set()
remaining_cases = set()
for v in video_annotations.keys():
    if video_annotations[v] == ["pseudo-segmentation"]:
        # cases that have no ground truth segmentation go to train set
        train_cases.add(v)
    elif "segmentation" in video_annotations[v]:
        # cases that have ground truth segmentation go to test/val set (as data is so scarce)
        test_val_cases.add(v)
    else:
        # remaining cases can be randomly assigned
        remaining_cases.add(v)


In [33]:
print(len(train_cases))
print(len(test_val_cases))
print(len(remaining_cases))

28
17
40


In [34]:
random.seed(seed)
more_test_val_cases = random.sample(list(remaining_cases), int(0.3 * len(remaining_cases)))
test_val_cases.update(more_test_val_cases)
remaining_cases.difference_update(more_test_val_cases)
train_cases.update(remaining_cases)

In [35]:
print(len(train_cases))
print(len(test_val_cases))
assert len(train_cases) + len(test_val_cases) == num_videos
assert len(train_cases.intersection(test_val_cases)) == 0

56
29


In [36]:
val_cases = random.sample(list(test_val_cases), int(0.5 * len(test_val_cases)))
test_cases = test_val_cases.difference(set(val_cases))

In [37]:

train_cases = sorted(list(train_cases))
val_cases = sorted(list(val_cases))
test_cases = sorted(list(test_cases))

In [ ]:
def write_cases_to_file(cases, filename):
    with open(os.path.join(path_data_cholec_split, filename), "w") as f:
        for case in cases:
            f.write(f"{case:02d}\n")

os.makedirs(path_data_cholec_split, exist_ok=False)
write_cases_to_file(train_cases, "cholec80_train.txt")
write_cases_to_file(val_cases, "cholec80_val.txt")
write_cases_to_file(test_cases, "cholec80_test.txt")
